In [26]:
from langgraph.graph import StateGraph, END, START
from typing import TypedDict, Annotated, Literal
import operator
import time

# Define the state schema
class AgentState(TypedDict):
    pick: int
    lnode: Annotated[list, operator.add]
    scratch: str
    count: Annotated[int, operator.add]

# Define node functions
def node1(state: AgentState):
    print("Start: ====== N1 ====== ")
    time.sleep(1)
    print("End: ====== N1 ====== ")
    return {"count": 1, "lnode": ["1"]}

def node2(state: AgentState):
    print("Start: ====== N2 ====== ")
    time.sleep(2)
    print("End: ====== N2 ====== ")
    return {"count": 1, "lnode": ["2"]}

def node3(state: AgentState):
    print("Start: ====== N3 ====== ")
    time.sleep(3)
    print("End: ====== N3 ====== ")
    return {"count": 1, "lnode": ["3"]}

def node4(state: AgentState):
    print("Start: ====== N4 ====== ")
    time.sleep(4)
    print("End: ====== N4 ====== ")
    return {"count": 1, "lnode": ["4"]}

def node5(state: AgentState):
    print("Start: ====== N5 ====== ")
    time.sleep(5)
    print("End: ====== N5 ====== ")
    return {"count": 1, "lnode": ["5"]}

def node6(state: AgentState):
    print("Start: ====== N6 ====== ")
    time.sleep(6)
    print("End: ====== N6 ====== ")
    return {"count": 1, "lnode": ["6"]}

# Decision function from Node1
def decide_next_node(state: AgentState) -> Literal["Node2", "Node3", "Node4"]:
    num = state['pick']
    print("Pick ===> {}".format(num))
    if num == 2:
        return "Node2"
    elif num == 3:
        return "Node3"
    else:
        return "Node4"

# Conditional logic for Node2 and Node3
def node2_next(state: AgentState):
    return "Node6" if "4" not in state["lnode"] else END

def node3_next(state: AgentState):
    return "Node6" if "4" not in state["lnode"] else END

# Build the graph
builder = StateGraph(AgentState)

# Add nodes
builder.add_node("Node1", node1)
builder.add_node("Node2", node2)
builder.add_node("Node3", node3)
builder.add_node("Node4", node4)
builder.add_node("Node5", node5)
builder.add_node("Node6", node6)

# Define entry point
builder.add_edge(START, "Node1")

# Node1 → (Node2, Node3, or Node4) depending on 'pick'
builder.add_conditional_edges("Node1", decide_next_node)

# Node4 → Node2 & Node3 (parallel)
builder.add_edge("Node4", "Node2")
builder.add_edge("Node4", "Node3")

# After both Node2 and Node3 (if triggered from Node4) → Node5
builder.add_edge(["Node2", "Node3"], "Node5")

# Conditional transitions from Node2 and Node3 to Node6 (only if not from Node4)
builder.add_conditional_edges("Node2", node2_next)
builder.add_conditional_edges("Node3", node3_next)

# Final transitions to END
builder.add_edge("Node5", END)
builder.add_edge("Node6", END)

# Compile the graph
graph = builder.compile()

In [24]:
state_done = graph.invoke({"count": 0, "scratch": "hi", "pick":2})
state_done

Start: ====== N1 ====== 
End: ====== N1 ====== 
Pick ===> 2
Start: ====== N2 ====== 
End: ====== N2 ====== 
Start: ====== N6 ====== 
End: ====== N6 ====== 


{'pick': 2, 'lnode': ['1', '2', '6'], 'scratch': 'hi', 'count': 3}

In [18]:
state_done = graph.invoke({"count": 0, "scratch": "hi", "pick":3})
state_done

Start: ====== N1 ====== 
End: ====== N1 ====== 
Pick ===> 3
Start: ====== N3 ====== 
End: ====== N3 ====== 
Start: ====== N6 ====== 
End: ====== N6 ====== 


{'pick': 3, 'lnode': ['1', '3', '6'], 'scratch': 'hi', 'count': 3}

In [27]:
state_done = graph.invoke({"count": 0, "scratch": "hi", "pick":4})
state_done

Start: ====== N1 ====== 
End: ====== N1 ====== 
Pick ===> 4
Start: ====== N4 ====== 
End: ====== N4 ====== 
Start: ====== N2 ====== 
Start: ====== N3 ====== 
End: ====== N2 ====== 
End: ====== N3 ====== 
Start: ====== N5 ====== 
End: ====== N5 ====== 


{'pick': 4, 'lnode': ['1', '4', '2', '3', '5'], 'scratch': 'hi', 'count': 5}